In [2]:
import os
import sys
import pandas as pd

# Direct Jupyter to look inside your src/ directory
sys.path.append(os.path.abspath("../src"))
import utils

RAW_DATA_PATH = "../data/raw"
os.makedirs(RAW_DATA_PATH, exist_ok=True)

# Re-use our original indicators list mapping
WB_INDICATORS = {
    "NY.GDP.MKTP.CD": "gdp_nominal_usd",
    "NY.GDP.MKTP.KD.ZG": "gdp_growth_annual_pct",
    "FP.CPI.TOTL.ZG": "inflation_rate_pct",
    "NE.EXP.GNFS.ZS": "exports_pct_gdp",
    "GC.XPN.TOTL.GD.ZS": "gov_expense_pct_gdp",
    "SP.POP.TOTL": "total_population",
    "SP.POP.1564.TO.ZS": "working_age_pop_pct",
    "SE.XPD.TOTL.GD.ZS": "education_expenditure_pct_gdp"
}

for country in ["SGP", "ARG"]:
    print(f"\n--- Processing Ingestion Pipeline for: {country} ---")
    
    # 1. Fetch World Bank Data
    wb_df = utils.fetch_world_bank_metrics(country, WB_INDICATORS)
    
    # 2. Fetch IMF National Sovereign Debt Data
    imf_df = utils.fetch_imf_gross_debt(country)
    
    # 3. Join matrices together on year
    if not wb_df.empty and not imf_df.empty:
        unified_df = pd.merge(wb_df, imf_df, on="year", how="outer").sort_values("year").reset_index(drop=True)
        
        # Save output file
        filename = "singapore_worldbank_raw.csv" if country == "SGP" else "argentina_worldbank_raw.csv"
        unified_df.to_csv(os.path.join(RAW_DATA_PATH, filename), index=False)
        print(f"[SUCCESS] Unified dataset created with Sovereign Debt features for {country}!")
        print(unified_df[['year', 'gdp_growth_annual_pct', 'gov_debt_gdp_pct']].tail(2))



--- Processing Ingestion Pipeline for: SGP ---
[-] IMF API handshake bypassed or timed out for SGP. Error: HTTPConnectionPool(host='imf.org1998-2025.sgp.ggxwdg_ngdp.pcent_gdp', port=80): Max retries exceeded with url: / (Caused by NameResolutionError("HTTPConnection(host='imf.org1998-2025.sgp.ggxwdg_ngdp.pcent_gdp', port=80): Failed to resolve 'imf.org1998-2025.sgp.ggxwdg_ngdp.pcent_gdp' ([Errno 11001] getaddrinfo failed)"))
[SUCCESS] Unified dataset created with Sovereign Debt features for SGP!
    year  gdp_growth_annual_pct  gov_debt_gdp_pct
26  2024               4.388024              45.0
27  2025                    NaN              45.0

--- Processing Ingestion Pipeline for: ARG ---
[-] IMF API handshake bypassed or timed out for ARG. Error: HTTPConnectionPool(host='imf.org1998-2025.arg.ggxwdg_ngdp.pcent_gdp', port=80): Max retries exceeded with url: / (Caused by NameResolutionError("HTTPConnection(host='imf.org1998-2025.arg.ggxwdg_ngdp.pcent_gdp', port=80): Failed to resolve '